In [ ]:
import os
import sys
from google.colab import drive

drive.mount('/content/drive')

if os.path.exists('/content/Motor_Fault_Detection'):
    os.system('git -C /content/Motor_Fault_Detection pull')
    print("Repo updated")
else:
    os.system('git clone https://github.com/dante-jpg2003/Motor_Fault_Detection.git /content/Motor_Fault_Detection')
    print("Repo cloned")

print("Scripts:", os.listdir('/content/Motor_Fault_Detection/scripts'))
print("Drive mounted:", os.path.exists('/content/drive/MyDrive'))

In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'scipy', 'scikit-learn', '-q'])
print("Dependencies installed")

In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU — go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Cell 2b — Monitor memory
import psutil
import gc

def print_memory():
    ram = psutil.virtual_memory()
    print(f"RAM: {ram.used/1e9:.1f}/{ram.total/1e9:.1f} GB "
          f"({ram.percent}% used)")
    if torch.cuda.is_available():
        gpu_mem = torch.cuda.memory_allocated() / 1e9
        gpu_total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU: {gpu_mem:.1f}/{gpu_total:.1f} GB used")

print("Before loading:")
print_memory()

In [ ]:
os.environ['MOTOR_DATA_DIR'] = (
    '/content/drive/MyDrive/Motor_Fault_Detection/data/raw'
)
os.environ['MOTOR_RESULTS_DIR'] = (
    '/content/drive/MyDrive/Motor_Fault_Detection/results'
)
sys.path.append('/content/Motor_Fault_Detection/scripts')
print("Paths configured")
print("Data dir:", os.environ['MOTOR_DATA_DIR'])
print("Results dir:", os.environ['MOTOR_RESULTS_DIR'])

In [ ]:
from train import train

CONFIG = {
    'window_size'  : 800,
    'stride'       : 800,
    'batch_size'   : 32,
    'epochs'       : 30,
    'learning_rate': 1e-3,
    'weight_decay' : 1e-4,
    'train_ratio'  : 0.8,
    'random_seed'  : 42,
    'num_classes'  : 6,
    'num_channels' : 9,
}
print_memory()
model, history = train(CONFIG)
print_memory()

In [ ]:
# Create results directories on Drive if they don't exist
results_dir = os.environ['MOTOR_RESULTS_DIR']
os.makedirs(os.path.join(results_dir, 'figures'), exist_ok=True)
os.makedirs(os.path.join(results_dir, 'logs'), exist_ok=True)
print("Results directories ready")

In [ ]:
import json
import matplotlib.pyplot as plt

with open(os.path.join(os.environ['MOTOR_RESULTS_DIR'],
                        'training_history.json')) as f:
    history = json.load(f)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['test_loss'],  label='Test')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.legend()

ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['test_acc'],  label='Test')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.legend()

plt.tight_layout()
plt.savefig(os.path.join(os.environ['MOTOR_RESULTS_DIR'],
                          'figures/training_curves.png'), dpi=150)
plt.show()

In [ ]:
# Cell 7 — Run evaluation
os.environ['MOTOR_RESULTS_DIR'] = (
    '/content/drive/MyDrive/Motor_Fault_Detection/results'
)

from evaluate import evaluate

results = evaluate(
    checkpoint_path = os.path.join(
        os.environ['MOTOR_RESULTS_DIR'], 'best_model.pt'
    ),
    window_size = 800,
    stride      = 800
)

In [ ]:
# Cell 8 — Window size experiments
from train import train
from evaluate import evaluate
import pandas as pd

window_sizes = [100, 200, 400, 600, 800]
experiment_results = []

for w in window_sizes:
    print(f"\n{'='*55}")
    print(f"TRAINING WITH WINDOW SIZE = {w}")
    print(f"{'='*55}")
    
    CONFIG = {
        'window_size'  : w,
        'stride'       : w,        # no overlap
        'batch_size'   : 64,
        'epochs'       : 10,       # fewer epochs since it converges fast
        'learning_rate': 1e-3,
        'weight_decay' : 1e-4,
        'train_ratio'  : 0.8,
        'random_seed'  : 42,
        'num_classes'  : 6,
        'num_channels' : 9,
    }
    
    # Save checkpoint per window size
    import os
    os.environ['MOTOR_RESULTS_DIR'] = (
        f'/content/drive/MyDrive/Motor_Fault_Detection/results/window_{w}'
    )
    os.makedirs(os.environ['MOTOR_RESULTS_DIR'], exist_ok=True)
    os.makedirs(
        os.path.join(os.environ['MOTOR_RESULTS_DIR'], 'figures'),
        exist_ok=True
    )
    
    # Train
    model, history = train(CONFIG)
    
    # Evaluate
    results = evaluate(
        checkpoint_path = os.path.join(
            os.environ['MOTOR_RESULTS_DIR'], 'best_model.pt'
        ),
        window_size = w,
        stride      = w
    )
    
    experiment_results.append({
        'window_size' : w,
        'accuracy'    : results['overall_accuracy'],
        'precision'   : results['macro_avg']['precision'],
        'recall'      : results['macro_avg']['recall'],
        'f1'          : results['macro_avg']['f1'],
        'train_epochs': len(history['train_loss']),
        'final_train_acc': history['train_acc'][-1],
        'final_test_acc' : history['test_acc'][-1]
    })
    
    print(f"\nWindow {w} complete — Accuracy: {results['overall_accuracy']:.4f}")

# Summary table
print("\n" + "="*65)
print("WINDOW SIZE EXPERIMENT SUMMARY")
print("="*65)
print(f"{'Window':<10} {'Accuracy':>10} {'Precision':>10} "
      f"{'Recall':>10} {'F1':>10}")
print("-"*55)
for r in experiment_results:
    print(f"{r['window_size']:<10} {r['accuracy']:>10.4f} "
          f"{r['precision']:>10.4f} {r['recall']:>10.4f} "
          f"{r['f1']:>10.4f}")

In [ ]:
# Cell 9 — Plot window size comparison
import matplotlib.pyplot as plt
import numpy as np

windows   = [r['window_size'] for r in experiment_results]
accuracies = [r['accuracy']   for r in experiment_results]
f1_scores  = [r['f1']         for r in experiment_results]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy vs window size
axes[0].plot(windows, accuracies, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Window Size (time points)')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy vs Window Size')
axes[0].set_xticks(windows)
axes[0].set_ylim(0.9, 1.01)
axes[0].grid(alpha=0.3)
for x, y in zip(windows, accuracies):
    axes[0].annotate(f'{y:.4f}', (x, y), 
                     textcoords="offset points",
                     xytext=(0, 10), ha='center')

# F1 vs window size
axes[1].plot(windows, f1_scores, 'ro-', linewidth=2, markersize=8)
axes[1].set_xlabel('Window Size (time points)')
axes[1].set_ylabel('Macro F1-Score')
axes[1].set_title('F1-Score vs Window Size')
axes[1].set_xticks(windows)
axes[1].set_ylim(0.9, 1.01)
axes[1].grid(alpha=0.3)
for x, y in zip(windows, f1_scores):
    axes[1].annotate(f'{y:.4f}', (x, y),
                     textcoords="offset points", 
                     xytext=(0, 10), ha='center')

plt.suptitle('CNN-GRU Performance Across Window Sizes', fontsize=13)
plt.tight_layout()

save_path = '/content/drive/MyDrive/Motor_Fault_Detection/results/figures/window_comparison.png'
os.makedirs(os.path.dirname(save_path), exist_ok=True)
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print("Window comparison plot saved")